# 使用注意力权重进行基因调控网络（GRN）推断

本教程演示如何利用scGPT Transformer模型中的注意力权重来推断基因调控网络。Transformer模型中的注意力机制可以揭示基因之间的潜在调控关系。


In [1]:
# 导入必要的库
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import torch
import matplotlib.pyplot as plt
import seaborn as sns

# 添加scGPT路径
sys.path.insert(0, "../")
import scgpt as scg
from scgpt.model import TransformerModel

print("✅ 所有库加载完成！")

In [2]:
# 设置参数
model_dir = Path("../save/scGPT_human")
data_path = Path("../data/GRN/attention_grn.h5ad")
save_dir = Path("../results/attention_grn")

# 创建保存目录
save_dir.mkdir(parents=True, exist_ok=True)

print("✅ 参数设置完成！")

In [3]:
# 加载数据
adata = sc.read_h5ad(data_path)

# 预处理
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

print(f"✅ 数据加载完成！形状: {adata.shape}")

In [4]:
# 加载预训练模型
model = TransformerModel.load_pretrained(model_dir)
model.eval()

# 获取设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("✅ 预训练模型加载完成！")

In [5]:
# 获取注意力权重
attention_maps = scg.tasks.get_attention_weights(
    adata,
    model,
    model_dir,
    batch_size=32,
    device=device,
)

# 平均所有层和头的注意力权重
mean_attention = np.mean(attention_maps, axis=(0, 1))  # 平均层和头

print(f"✅ 注意力权重获取完成！形状: {attention_maps.shape}")
print(f"平均注意力形状: {mean_attention.shape}")

In [6]:
# 分析注意力模式
genes = adata.var.index.tolist()

# 1. 找出高注意力基因对
threshold = 0.1  # 注意力权重阈值
high_attention_pairs = []

for i in range(len(genes)):
    for j in range(len(genes)):
        if i != j and mean_attention[i, j] > threshold:
            high_attention_pairs.append({
                "source": genes[i],
                "target": genes[j],
                "attention": float(mean_attention[i, j])
            })

# 按注意力权重排序
high_attention_pairs.sort(key=lambda x: x["attention"], reverse=True)

print(f"\n高注意力基因对（前10个）:")
for pair in high_attention_pairs[:10]:
    print(f"  {pair['source']} -> {pair['target']}: {pair['attention']:.4f}")

# 2. 可视化注意力热图
plt.figure(figsize=(12, 10))
sns.heatmap(
    mean_attention[:30, :30],
    xticklabels=genes[:30],
    yticklabels=genes[:30],
    cmap="viridis",
    cbar=True,
    vmin=0,
    vmax=0.3
)
plt.title("基因注意力权重热图（前30个基因）")
plt.xlabel("目标基因")
plt.ylabel("源基因")
plt.xticks(rotation=45)
plt.yticks(rotation=45)
plt.tight_layout()
plt.show()

print("✅ 注意力分析完成！")

In [7]:
# 保存结果
# 1. 保存注意力权重矩阵
np.save(save_dir / "attention_weights.npy", mean_attention)

# 2. 保存高注意力基因对
pd.DataFrame(high_attention_pairs).to_csv(
    save_dir / "high_attention_pairs.csv",
    index=False
)

print(f"✅ 结果已保存！共发现 {len(high_attention_pairs)} 对高注意力基因")